In [7]:
import pandas as pd
import re

INPUT_FILE = "education.csv"
OUTPUT_FILE = "formatted_education.csv"
DOMAIN = "education"
TOP_N = 5

df = pd.read_csv(INPUT_FILE)

df["O*NET-SOC Code"] = df["O*NET-SOC Code"].astype(str).str.strip()
df["Title"] = df["Title"].astype(str).str.strip()
df["Scale ID"] = df["Scale ID"].astype(str).str.strip()
df["Scale Name"] = df["Scale Name"].astype(str).str.strip()
df["Element Name"] = df["Element Name"].astype(str).str.strip()
df["Category"] = pd.to_numeric(df["Category"], errors="coerce")
df["Data Value"] = pd.to_numeric(df["Data Value"], errors="coerce")

df["attribute"] = df["Category"].apply(
    lambda x: f"category_{int(x)}" if pd.notna(x) else None
)

def clean_scale(scale_id, scale_name):
    if scale_id == "RL":
        return "required_level"
    elif scale_id == "IM":
        return "importance"
    elif scale_id == "RQ":
        return "required_quantity"
    else:
        name = str(scale_name).lower()
        name = re.sub(r"[^a-z0-9]+", "_", name)
        return name.strip("_")

df["scale"] = df.apply(
    lambda row: clean_scale(row["Scale ID"], row["Scale Name"]),
    axis=1
)

df["output_column"] = (
    DOMAIN
    + "_"
    + df["attribute"]
    + "_"
    + df["scale"]
    + "_value"
)

wide = df.pivot_table(
    index=["O*NET-SOC Code", "Title"],
    columns="output_column",
    values="Data Value",
    aggfunc="first"
).reset_index()

wide.columns.name = None

ranked_columns = []

for scale in df["scale"].dropna().unique():
    scale_df = df[df["scale"] == scale].copy()

    scale_df = scale_df.dropna(
        subset=["attribute", "Data Value"]
    )

    scale_df = scale_df.sort_values(
        ["O*NET-SOC Code", "Data Value"],
        ascending=[True, False]
    )

    ranked = (
        scale_df
        .groupby("O*NET-SOC Code")["attribute"]
        .apply(
            lambda x: "|".join(
                x.drop_duplicates().head(TOP_N)
            )
        )
        .reset_index()
    )

    ranked_column_name = f"{DOMAIN}_{scale}_ranked"

    ranked = ranked.rename(
        columns={"attribute": ranked_column_name}
    )

    wide = wide.merge(
        ranked,
        on="O*NET-SOC Code",
        how="left"
    )

    ranked_columns.append(ranked_column_name)

base_columns = [
    "O*NET-SOC Code",
    "Title"
]

other_columns = [
    col
    for col in wide.columns
    if col not in base_columns + ranked_columns
]

wide = wide[
    base_columns
    + ranked_columns
    + other_columns
]

wide = wide.sort_values(
    "O*NET-SOC Code"
).reset_index(drop=True)

wide.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Formatted shape:", wide.shape)
print("Number of occupations:", wide["O*NET-SOC Code"].nunique())
print("Ranked columns:", ranked_columns)
print(f"Saved to: {OUTPUT_FILE}")

Formatted shape: (901, 29)
Number of occupations: 901
Ranked columns: ['education_required_level_ranked', 'education_importance_ranked', 'education_required_quantity_ranked']
Saved to: formatted_education.csv


In [ ]:
import pandas as pd
import re

INPUT_FILE = "training_and_experience.csv"
OUTPUT_FILE = "formatted_training_and_experience.csv"
DOMAIN = "training_and_experience"
TOP_N = 5

df = pd.read_csv(INPUT_FILE)

df["O*NET-SOC Code"] = df["O*NET-SOC Code"].astype(str).str.strip()
df["Title"] = df["Title"].astype(str).str.strip()
df["Scale ID"] = df["Scale ID"].astype(str).str.strip()
df["Scale Name"] = df["Scale Name"].astype(str).str.strip()
df["Element Name"] = df["Element Name"].astype(str).str.strip()
df["Category"] = pd.to_numeric(df["Category"], errors="coerce")
df["Data Value"] = pd.to_numeric(df["Data Value"], errors="coerce")

def clean_name(name):
    name = str(name).lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

df["attribute"] = df["Element Name"].apply(clean_name)

def clean_scale(scale_id, scale_name):
    if scale_id == "RW":
        return "related_work_experience"
    elif scale_id == "PT":
        return "on_site_training"
    elif scale_id == "OJ":
        return "on_the_job_training"
    elif scale_id == "IM":
        return "importance"
    else:
        return clean_name(scale_name)

df["scale"] = df.apply(
    lambda row: clean_scale(row["Scale ID"], row["Scale Name"]),
    axis=1
)

df["output_column"] = (
    DOMAIN
    + "_"
    + df["attribute"]
    + "_"
    + df["scale"]
    + "_value"
)

wide = df.pivot_table(
    index=["O*NET-SOC Code", "Title"],
    columns="output_column",
    values="Data Value",
    aggfunc="first"
).reset_index()

wide.columns.name = None

ranked_columns = []

for scale in df["scale"].dropna().unique():
    scale_df = df[df["scale"] == scale].copy()

    scale_df = scale_df.dropna(
        subset=["attribute", "Data Value"]
    )

    scale_df = scale_df.sort_values(
        ["O*NET-SOC Code", "Data Value"],
        ascending=[True, False]
    )

    ranked = (
        scale_df
        .groupby("O*NET-SOC Code")["attribute"]
        .apply(
            lambda x: "|".join(
                x.drop_duplicates().head(TOP_N)
            )
        )
        .reset_index()
    )

    ranked_column_name = f"{DOMAIN}_{scale}_ranked"

    ranked = ranked.rename(
        columns={"attribute": ranked_column_name}
    )

    wide = wide.merge(
        ranked,
        on="O*NET-SOC Code",
        how="left"
    )

    ranked_columns.append(ranked_column_name)

base_columns = [
    "O*NET-SOC Code",
    "Title"
]

other_columns = [
    col
    for col in wide.columns
    if col not in base_columns + ranked_columns
]

wide = wide[
    base_columns
    + ranked_columns
    + other_columns
]

wide = wide.sort_values(
    "O*NET-SOC Code"
).reset_index(drop=True)

wide.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Formatted shape:", wide.shape)
print("Number of occupations:", wide["O*NET-SOC Code"].nunique())
print("Ranked columns:", ranked_columns)
print(f"Saved to: {OUTPUT_FILE}")